# RunStore + Replay Demo

Illustrates persisting a run and loading it for replay/audit using `RunStore`.

In [ ]:
import os
from getpass import getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

from agentic_codex import AgentBuilder, Context, GraphRunner, GraphNodeSpec, FunctionAdapter, EnvOpenAIAdapter, RunStore
from agentic_codex.core.schemas import AgentStep, Message

def stub_llm(prompt: str) -> str:
    return f"[stub llm] {prompt[:64]}"

try:
    llm_adapter = EnvOpenAIAdapter(model="gpt-4o-mini")
except Exception:
    llm_adapter = FunctionAdapter(stub_llm)

def make_agent(name: str):
    def step(ctx: Context) -> AgentStep:
        msg = f"{name}:{ctx.goal}"
        return AgentStep(out_messages=[Message(role=name, content=msg)], state_updates={name: True})
    return AgentBuilder(name=name, role=name).with_llm(llm_adapter).with_step(step).build()

a = make_agent("alpha")
b = make_agent("beta")

graph = {
    "alpha": GraphNodeSpec(agent=a),
    "beta": GraphNodeSpec(agent=b, after=["alpha"]),
}

runner = GraphRunner(graph, allow_parallel=False)
run_result = runner.run(goal="demo run", inputs={})

store = RunStore("runs/")
path = store.save(run_result)
replayed = store.load(run_result.run_id)

path, list(replayed.get("meta", {}).keys()), len(replayed.get("events", []))